# NPCC simulation study — results

This notebook analyzes the packaged conditional-copula simulation study
(`npcc.experiments`). Each estimator is scored against exact **pyvinecopulib**
ground truth — fixed-`x`, UV-integrated **IAE / ISE / KL** on `pdf`, `hfunc1`,
`hfunc2` — with margin/dependence diagnostics and plot-ready true-vs-predicted
density surfaces.

## How the study was run

The run is orchestrated by the CLI and is **memory-light** (~1.5 GB VRAM,
~3 GB RAM measured), so it runs on any modest GPU. Use the packaged script on
a GPU box — it checkpoints every cell, so it is safe to interrupt and resume:

```bash
# on a GPU box; CUDA_EXTRA matches the driver: cu126 | cu128 | cu130 | cu132
TABPFN_TOKEN=<token> CUDA_EXTRA=cu128 scripts/run_study.sh
# re-run the same command to resume after any interruption
```

or the CLI directly:

```bash
uv run npcc-simstudy --config configs/study.toml --out results/study \
  --workers 1 --resume --gpu-mem-fraction 0.9
```

The shipped `configs/study.toml` sweeps 2 families x 2 tau-scenarios x
n in {200, 500, 1000, 2000} x 10 reps over 14 estimators; its header says
how to time one cell before committing to the full run. Copy the
resulting `results/study/` next to this notebook; everything below just loads
it — **no GPU needed to analyze**.

## Load the saved results

`results/study/` holds the aggregated paper tables plus raw
`quantities`/`diagnostics`/`runtime` and per-cell checkpoints. The raw
per-repetition metrics live in the per-cell shards, so we concatenate them for
the distribution plots.

In [ ]:
import glob
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from npcc.experiments.plots import metric_boxplot_by_n

RESULTS = Path("../results/study")

selection = pd.read_parquet(RESULTS / "selection_summary.parquet")
summary_over_x = pd.read_parquet(RESULTS / "summary_over_x.parquet")
runtime_df = pd.read_parquet(RESULTS / "runtime.parquet")
quantity_df = pd.read_parquet(RESULTS / "quantities.parquet")

# Raw per-rep metrics: reassemble from the per-cell checkpoints.
metric_df = pd.concat(
  (
    pd.read_parquet(p)
    for p in sorted(glob.glob(str(RESULTS / "cells" / "*" / "metrics.parquet")))
  ),
  ignore_index=True,
)

cfg = json.loads((RESULTS / "config.json").read_text())["grid"]
print(
  f"{len(metric_df):,} metric rows | {metric_df['label'].nunique()} estimators"
  f" | families={cfg['families']} | n={cfg['n']} x {cfg['n_rep']} reps"
)

## Headline: which estimator wins

Average pdf-KL per estimator across every `(family, tau_scenario, n)` cell —
lower is better. (KL is the density-estimation target; by Rosenblatt it is the
copula-density objective.)

In [ ]:
board = (
  summary_over_x[
    (summary_over_x["quantity"] == "pdf") & (summary_over_x["metric"] == "KL")
  ]
  .groupby("label", as_index=False)["rep_mean"]
  .mean()
  .sort_values("rep_mean")
  .rename(columns={"rep_mean": "mean_pdf_KL"})
  .reset_index(drop=True)
)
board.style.format({"mean_pdf_KL": "{:.4f}"})

## Metric table

pdf-KL per `(family, tau_scenario, n)`, one column per estimator (`label`
already encodes backend + transform + hyperparameters). Swap `quantity` /
`metric` for `hfunc1`/`hfunc2` or `IAE`/`ISE`.

In [ ]:
pdf_kl = summary_over_x[
  (summary_over_x["quantity"] == "pdf") & (summary_over_x["metric"] == "KL")
]
table = pdf_kl.pivot_table(
  index=["family", "tau_scenario", "n"], columns="label", values="rep_mean"
)
table.style.format(precision=4)

## Distribution across repetitions

Per-rep pdf-KL by sample size for a representative subset of estimators (the
`label=` selector keeps the plot readable; drop it to show all 16).

In [ ]:
metric_boxplot_by_n(
  metric_df,
  family="clayton",
  tau_scenario="linear",
  quantity="pdf",
  metric="KL",
  label=[
    "tabpfn-quantiles-v3-logit",
    "lightgbm-logit",
    "nori-logit",
    "tabicl-logit",
    "xgboost-logit",
  ],
)

## PDF surfaces

True vs fitted conditional copula density at one Kendall's-tau slice (surfaces
are stored for the `surface_families`, i.e. clayton). The fitted surface is the
per-`(u, v)` median across repetitions, for the best estimator overall.

In [ ]:
FAMILY, SCENARIO, N, TARGET_TAU = "clayton", "linear", 1000, 0.5
LABEL = board.iloc[0]["label"]

surface = quantity_df[
  (quantity_df["family"] == FAMILY)
  & (quantity_df["tau_scenario"] == SCENARIO)
  & (quantity_df["quantity"] == "pdf")
  & (quantity_df["normalize"] == "none")
  & (quantity_df["n"] == N)
  & (quantity_df["target_tau"] == TARGET_TAU)
  & (quantity_df["label"] == LABEL)
]
surface = surface.groupby(["u", "v"], as_index=False).agg(
  truth=("truth", "first"), pred=("pred", "median")
)

fig = plt.figure(figsize=(11, 4.5))
for i, (column, title) in enumerate(
  [("truth", "True pdf"), ("pred", f"Estimated pdf ({LABEL})")], start=1
):
  ax = fig.add_subplot(1, 2, i, projection="3d")
  ax.plot_trisurf(
    surface["u"], surface["v"], surface[column], cmap="viridis", linewidth=0.1
  )
  ax.set_title(title)
  ax.set_xlabel("u")
  ax.set_ylabel("v")
  ax.set_zlabel("density")
fig.suptitle(f"pdf at target tau={TARGET_TAU} ({FAMILY}/{SCENARIO}, n={N})")
fig.tight_layout()
plt.show()

## Cost and memory footprint

Mean fit / total time and peak memory per estimator. This is why the study runs
on any modest GPU — peak VRAM is ~1.5 GB and RSS ~3 GB.

In [ ]:
cost = (
  runtime_df.groupby("label", as_index=False)
  .agg(
    fit_s=("fit_time", "mean"),
    total_s=("total_estimator_time", "mean"),
    gpu_peak_MiB=("gpu_peak_reserved_mb", "max"),
    rss_MiB=("rss_mb", "max"),
  )
  .sort_values("total_s", ascending=False)
  .reset_index(drop=True)
)
cost.style.format(
  {
    "fit_s": "{:.1f}",
    "total_s": "{:.1f}",
    "gpu_peak_MiB": "{:.0f}",
    "rss_MiB": "{:.0f}",
  }
)